# Saudi Real Estate Market Analysis
This notebook reproduces the main exploratory analysis of listing asking prices in Riyadh, Jeddah, Dammam, Khobar and Madinah.

**Data:** Upload `sa_listings_cleaned.csv` (Kaggle real-estate listings). **Interpretation:** Listings, not completed transactions. Screening thresholds are exploratory; their effect should be reviewed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Run this cell in Google Colab to upload the Kaggle CSV, if needed.
try:
    from google.colab import files
    if not Path('sa_listings_cleaned.csv').exists():
        files.upload()
except ImportError:
    pass

df = pd.read_csv('sa_listings_cleaned.csv')
print('All data:', df.shape)
df.head()

In [ ]:
# Initial inspection
print(df.info())
print('Missing values:\n', df.isna().sum().sort_values(ascending=False))
print('Repeated complete rows:', df.duplicated().sum())
print('Columns:', df.columns.tolist())

In [ ]:
cities = ['Riyadh', 'Jeddah', 'Dammam', 'Khobar', 'Madinah']
df_cities = df[df['city_en'].isin(cities)].copy()
print('City listings:', df_cities.shape)
print('Repeated listing IDs:', df_cities['id'].duplicated().sum())
print(df_cities['transaction_type'].value_counts(dropna=False))

In [ ]:
# Sales and separate property categories
df_sale = df_cities[(df_cities['transaction_type'] == 'Sale') & (df_cities['price'] > 1)].copy()
df_villas = df_sale[df_sale['property_category'] == 'Villa'].copy()
df_apartments = df_sale[df_sale['property_category'] == 'Apartment'].copy()

# Exploratory filtering rules used in the original analysis
df_villas_clean = df_villas[df_villas['area'] >= 100].copy()
df_apartments_clean = df_apartments[(df_apartments['price'] >= 10000) & (df_apartments['area'] >= 30)].copy()
for subset in [df_villas_clean, df_apartments_clean]:
    subset['price_per_sqm'] = subset['price'] / subset['area']
print('Villas:', df_villas_clean.shape, 'Apartments:', df_apartments_clean.shape)

In [ ]:
# City-level summaries
villa_summary = df_villas_clean.groupby('city_en').agg(villa_count=('price','count'),median_price=('price','median'),median_price_per_sqm=('price_per_sqm','median')).round(2)
apartment_summary = df_apartments_clean.groupby('city_en').agg(apartment_count=('price','count'),median_price=('price','median'),median_price_per_sqm=('price_per_sqm','median')).round(2)
comparison = pd.DataFrame({'Villa':villa_summary['median_price_per_sqm'],'Apartment':apartment_summary['median_price_per_sqm']})
comparison['difference_percent'] = ((comparison['Villa'] - comparison['Apartment']) / comparison['Apartment'] * 100)
display(comparison.round(2))

In [ ]:
# Price plots
for values, title, filename in [
    (villa_summary['median_price'], 'Median Villa Asking Prices by City', 'villa_prices_by_city.png'),
    (villa_summary['median_price_per_sqm'], 'Median Villa Asking Price per sqm', 'villa_price_per_sqm.png'),
    (apartment_summary['median_price_per_sqm'], 'Median Apartment Asking Price per sqm', 'apartment_price_per_sqm.png')]:
    ax = values.sort_values(ascending=False).plot.bar(figsize=(9,5), title=title)
    ax.set_ylabel('SAR')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()
ax = comparison[['Villa','Apartment']].plot.bar(figsize=(10,5),title='Villa vs Apartment Median Asking Price per sqm')
ax.set_ylabel('SAR per sqm')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('villa_vs_apartment.png', dpi=300)
plt.show()

In [ ]:
# Riyadh district comparison: at least 100 villa listings per district
df_riyadh = df_villas_clean[df_villas_clean['city_en'] == 'Riyadh'].copy()
district_prices = df_riyadh.groupby('district_en').agg(villa_count=('price','count'),median_price_per_sqm=('price_per_sqm','median'))
district_prices = district_prices[district_prices['villa_count'] >= 100]
top_districts = district_prices.nlargest(10,'median_price_per_sqm')
display(top_districts.round(2))
top_districts['median_price_per_sqm'].sort_values().plot.barh(figsize=(10,6),title='Top Riyadh Districts: Median Villa Asking Price per sqm')
plt.xlabel('SAR per sqm')
plt.tight_layout()
plt.savefig('riyadh_top_districts.png',dpi=300)
plt.show()

In [ ]:
# Area versus price (focused views use different ranges by property type)
df_villas_plot = df_villas_clean[(df_villas_clean['area'] <= 1000) & (df_villas_clean['price'] <= 10000000)].copy()
df_apartments_plot = df_apartments_clean[(df_apartments_clean['area'] <= 400) & (df_apartments_clean['price'] <= 2000000)].copy()
correlation_summary = pd.DataFrame({'Property Type':['Villa','Apartment'], 'Correlation':[df_villas_plot['area'].corr(df_villas_plot['price']),df_apartments_plot['area'].corr(df_apartments_plot['price'])]})
for subset, label, filename in [(df_villas_plot,'Villa','villa_area_vs_price.png'),(df_apartments_plot,'Apartment','apartment_area_vs_price_focused.png')]:
    plt.figure(figsize=(9,5))
    plt.scatter(subset['area'], subset['price'],alpha=.2,s=5)
    plt.title(f'{label} Area vs Asking Price — Focused View')
    plt.xlabel('Area (sqm)')
    plt.ylabel('Asking Price (SAR)')
    plt.tight_layout()
    plt.savefig(filename,dpi=300)
    plt.show()
display(correlation_summary.round(3))
correlation_summary.plot.bar(x='Property Type',y='Correlation',legend=False,title='Area vs Price Pearson Correlation',figsize=(8,5))
plt.ylim(0,1)
plt.tight_layout()
plt.savefig('property_correlation.png',dpi=300)
plt.show()

In [ ]:
# Annual apartment asking rents
rent = df_cities[(df_cities['transaction_type'] == 'Rent') & (df_cities['rent_period'] == 'Yearly') & (df_cities['property_category'] == 'Apartment')].copy()
df_rent_clean = rent[rent['price'] >= 5000].copy()
df_rent_area_clean = df_rent_clean[df_rent_clean['area'].between(30,500)].copy()
rent_summary = df_rent_clean.groupby('city_en').agg(apartment_count=('price','count'),median_annual_rent=('price','median'),mean_annual_rent=('price','mean')).round(2)
display(rent_summary)
ax = rent_summary['median_annual_rent'].sort_values(ascending=False).plot.bar(figsize=(9,5), title='Median Annual Apartment Asking Rent by City')
ax.set_ylabel('SAR per year')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('annual_apartment_rent.png',dpi=300)
plt.show()

In [ ]:
# Export summary data
final_summary = pd.DataFrame({'Villa Price per sqm':villa_summary['median_price_per_sqm'],'Apartment Price per sqm':apartment_summary['median_price_per_sqm'],'Annual Apartment Rent':rent_summary['median_annual_rent']}).round(2)
for filename, data in [('villa_summary.csv',villa_summary),('apartment_summary.csv',apartment_summary),('property_comparison.csv',comparison),('annual_rent_summary.csv',rent_summary),('final_real_estate_summary.csv',final_summary)]:
    data.to_csv(filename)
correlation_summary.to_csv('correlation_summary.csv',index=False)
display(final_summary)
print('Summary CSV files and charts saved to the notebook working directory.')

## Limitations
- The dataset consists of asking-price listings, not verified transaction records.
- The year and representativeness of data require independent verification.
- Small and extremely large areas/prices may contain data-entry errors.
- Sale vs rent and property category were separated intentionally.
- Rental comparison summarizes price-screened yearly listings, while the area-screened rental subset is used only for area-related work.
- Correlation is descriptive, does not establish causality, and focused subsets have different range restrictions.